# Matched vs Legacy Canary Sweep v2 (holdout estimator; sigma = 1.1, 3.0, 5.0)

**Goal:** replace the pathological high-sigma canary results (244%/657% tightness) with honest numbers from the
**matched (exchangeable) canary design**, and quantify the appearance artifact via the legacy-vs-matched delta.

**v2 (2026-07-08):** primary estimate now uses the selection-valid **sample-split holdout** Wilson bound
(2026-07-06 estimator fix); the legacy in-sample value is retained per row as a diagnostic
(`epsilon_lower_conservative_insample`). Pre-flight cells ABORT the run if the uploaded bundle is stale.

**Runtime:** GPU required (T4 OK). Full sweep = 33 DP-SGD trainings x 15 epochs, roughly **2.5-4 h**.
Tip: run the smoke test first (2 min) to validate, then the full sweep.

**Steps:** (1) upload `matched_canary_bundle_v2.zip`, (2) Run all, (3) download `matched_canary_results_v2.zip` into `codex/results/`.

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload matched_canary_bundle_v2.zip
import zipfile, os
zipfile.ZipFile('matched_canary_bundle_v2.zip').extractall('project')
os.chdir('project')
print(sorted(os.listdir('.')))

In [ ]:
%pip -q install opacus dp-accounting pyyaml pytest scipy
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU ONLY - fix runtime type!')

In [ ]:
# --- PRE-FLIGHT 1/3: bundled estimator test suite (aborts on any failure) ----
# 13 tests incl. HoldoutThresholdSelectionTests null coverage (200 pure-noise
# reps -> holdout conservative eps must be 0 in >=95%).
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "unittest", "tests.test_empirical", "-v"],
                   capture_output=True, text=True)
print(r.stderr[-3000:])
assert r.returncode == 0, "TEST SUITE FAILED -- do NOT run the experiment."
print("Bundled estimator test suite: 13/13 PASS")

In [ ]:
# --- PRE-FLIGHT 2/3: v2 fix markers + estimator sanity (aborts) --------------
import sys, random, inspect
sys.path.insert(0, "src"); sys.path.insert(0, "codex")
from dp_audit_tightness.privacy.empirical import estimate_empirical_lower_bound
from dp_audit_tightness.privacy.pld_accounting import compute_epsilon_pld
import run_matched_canary_sweep as runner

checks = []
# (a) holdout path exists, is wired as PRIMARY in estimate_row, and works
sig = inspect.signature(estimate_empirical_lower_bound)
checks.append(("estimator exposes threshold_selection", "threshold_selection" in sig.parameters))
er = inspect.getsource(runner.estimate_row)
checks.append(("estimate_row primary = holdout", 'threshold_selection="holdout"' in er))
checks.append(("estimate_row keeps in-sample as diagnostic", "epsilon_lower_conservative_insample" in er))
random.seed(608)
kw = dict(delta=1e-5, align_event_to_score_direction=True, require_member_favoring=True,
          report_confidence_supported_lower_bound=True, score_direction="higher")
m = [random.gauss(0, 1) for _ in range(640)]; n = [random.gauss(0, 1) for _ in range(640)]
e0 = estimate_empirical_lower_bound(member_scores=m, nonmember_scores=n,
                                    threshold_selection="holdout", **kw)
checks.append(("holdout null -> eps=0", e0.epsilon_lower_empirical_conservative == 0.0))
checks.append(("holdout method tag", e0.threshold_selection == "holdout"))
m = [random.gauss(3, 1) for _ in range(640)]; n = [random.gauss(0, 1) for _ in range(640)]
e1 = estimate_empirical_lower_bound(member_scores=m, nonmember_scores=n,
                                    threshold_selection="holdout", **kw)
checks.append(("holdout signal -> eps>0", e1.epsilon_lower_empirical_conservative > 0.5))
# (b) matched design registered alongside legacy
checks.append(("matched strategy wired", runner.DESIGNS.get("matched") == "matched_random_canaries"))
checks.append(("legacy strategy wired", runner.DESIGNS.get("legacy") == "random_canaries"))
# (c) hard PLD backend: default google AND dp_accounting importable + true-PLD value
checks.append(("compute_epsilon_pld default backend=google",
               inspect.signature(compute_epsilon_pld).parameters["backend"].default == "google"))
res = compute_epsilon_pld(noise_multiplier=1.1, sampling_rate=128/2048, num_steps=16, delta=1e-5)
checks.append(("PLD backend is google_dp_accounting", res["backend_used"] == "google_dp_accounting"))

failed = [name for name, ok in checks if not ok]
for name, ok in checks: print(("PASS " if ok else "FAIL ") + name)
if failed:
    raise SystemExit(f"SANITY CHECKS FAILED: {failed} -- do NOT run the experiment.")
print("\nAll v2 pre-flight checks passed -- safe to run.")

In [ ]:
# --- PRE-FLIGHT 3/3: matched-design unit tests must pass before burning GPU hours.
!python -m pytest tests/test_matched_canaries.py -q

In [ ]:
# Smoke run (~2 min on GPU): 1 sigma, 2 seeds, 1 epoch, 100 canaries/group.
# Verifies the whole path end-to-end. Expect NO pathological rows.
!SMOKE=1 python codex/run_matched_canary_sweep.py

In [ ]:
# FULL sweep (~2.5-4 h). 3 sigmas x (1 base + 2 designs x 5 audit seeds) trainings, 15 epochs each.
!python codex/run_matched_canary_sweep.py

In [ ]:
import json, pandas as pd
rows = json.load(open('codex/results/matched_canary_sweep/matched_canary_summary.json'))
df = pd.DataFrame(rows)
cols = ['sigma','design','threshold_selection','epsilon_lower_conservative',
        'epsilon_lower_conservative_insample','epsilon_lower_gdp','epsilon_upper_pld',
        'tightness_pld','pathological','member_score_mean','nonmember_score_mean','warning']
display(df[cols])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
fig, ax = plt.subplots(figsize=(7,4.5))
for design, marker in [('legacy','o'),('matched','s')]:
    sub = df[df.design==design].sort_values('sigma')
    ax.plot(sub.sigma, sub.epsilon_lower_conservative, marker+'-', label=f'{design} eps_lower (holdout)')
sub = df[df.design=='matched'].sort_values('sigma')
ax.plot(sub.sigma, sub.epsilon_upper_pld, 'r:', label='eps_upper (PLD)')
ax.set_xlabel('noise multiplier sigma'); ax.set_ylabel('epsilon')
ax.set_title('Legacy vs matched canary design: the appearance artifact (holdout estimator)')
ax.legend(); plt.tight_layout()
plt.savefig('codex/results/matched_canary_sweep/legacy_vs_matched.png', dpi=150)
plt.show()

In [ ]:
!cd codex/results && zip -r ../../matched_canary_results_v2.zip matched_canary_sweep
from google.colab import files
files.download('matched_canary_results_v2.zip')